# Lab 0 Task 0.2.2 — Transfer Learning from MNIST to SVHN

This notebook implements Task 0.2.2:

1. **Experiment A** — Train a `SimpleCNN` on MNIST from scratch; report test accuracy.
2. **Experiment B** — Evaluate the MNIST-trained model directly on SVHN (zero-shot transfer); report test accuracy.
3. **Experiment C** — Fine-tune the MNIST-trained model on SVHN (transfer learning); report test accuracy.

The architecture mirrors `SimpleCNN` from `Lab0_01`, with `in_channels=3` throughout.  
MNIST images (grayscale) are converted to pseudo-RGB by replicating the single channel three times, so all weights transfer cleanly between experiments.

## Setup & Imports

In [1]:
import copy
import warnings
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torchvision.datasets import MNIST, SVHN
from utils import evaluate, fit, make_loaders

# Suppress NumPy 2.4 VisibleDeprecationWarning triggered inside torchvision
warnings.filterwarnings("ignore", category=np.exceptions.VisibleDeprecationWarning)

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Make results reproducible
torch.manual_seed(1)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(1)

2.11.0+cu130
True
NVIDIA GeForce RTX 2080 Ti
Using device: cuda


## Notebook parameters

In [2]:
LOG_WANDB   = True  # Set False to disable wandb logging
NUM_WORKERS = 8
PIN_MEMORY  = True

## Model Definition

We adapt `SimpleCNN` from Lab 0.1 for 32 × 32 images. The architecture is identical; only `in_channels=3` is fixed.

| Layer | Detail |
|---|---|
| Conv Block 1 | Conv2d(3 → 32, 3×3, pad 1) → LeakyReLU → MaxPool2d |
| Conv Block 2 | Conv2d(32 → 64, 3×3, pad 1) → LeakyReLU → MaxPool2d |
| Conv Block 3 | Conv2d(64 → 128, 3×3, pad 1) → LeakyReLU → MaxPool2d |
| FC 1 | Linear(128 × 4 × 4 → 256) → LeakyReLU |
| FC 2 | Linear(256 → `num_classes`) |

In [3]:
class SimpleCNN(nn.Module):
    """
    Adapted SimpleCNN for 3-channel input and 32×32 images.

    Architecture (identical to Lab0_01):
        Conv Block 1 : Conv2d(3  → 32)  → LeakyReLU → MaxPool2d  (32→16)
        Conv Block 2 : Conv2d(32 → 64)  → LeakyReLU → MaxPool2d  (16→ 8)
        Conv Block 3 : Conv2d(64 → 128) → LeakyReLU → MaxPool2d  ( 8→ 4)
        FC 1         : Linear(128*4*4 → 256) → LeakyReLU
        FC 2         : Linear(256 → num_classes)
    """

    def __init__(
        self,
        num_classes: int = 10,
        activation: nn.Module = nn.LeakyReLU(),
    ) -> None:
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3,  32,  kernel_size=3, padding=1),
            activation,
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            activation,
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            activation,
            nn.MaxPool2d(2, 2),
        )

        self.classifier = nn.Sequential(
            nn.Linear(128 * 4 * 4, 256),
            activation,
            nn.Linear(256, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = torch.flatten(x, start_dim=1)
        return self.classifier(x)

## Data Loading

### MNIST
- Grayscale, 28 × 28 → resized to 32 × 32 (match SVHN's native resolution)
- Converted to pseudo-RGB via `Grayscale(num_output_channels=3)` — the single channel is replicated across all three channels
- Normalised to mean 0.5 per channel, std 0.5 per channel

### SVHN
- RGB, 32 × 32 (no resizing needed)
- Normalised to mean 0.5 per channel, std 0.5 per channel

Both datasets produce identical tensor shapes `[3, 32, 32]`, so the same `SimpleCNN` is used across all experiments and all weights transfer cleanly.

In [4]:
BATCH_SIZE   = 256
VAL_FRACTION = 0.2  # 20% of each training split held out for validation

loader_kwargs = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

# ── MNIST ──────────────────────────────────────────────────────────────────
mnist_transform = transforms.Compose([
    transforms.Resize((32, 32)),                  # match SVHN spatial resolution
    transforms.Grayscale(num_output_channels=3),  # replicate channel: [1,H,W] → [3,H,W]
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

mnist_train = MNIST(root="../data", train=True,  download=True, transform=mnist_transform)
mnist_test  = MNIST(root="../data", train=False, download=True, transform=mnist_transform)

mnist_train_loader, mnist_val_loader, mnist_test_loader = make_loaders(
    mnist_train, mnist_test,
    loader_kwargs=loader_kwargs,
    val_fraction=VAL_FRACTION,
)
print(f"MNIST classes: {mnist_train.classes}")

Dataset split — train: 48,000  val: 12,000  test: 10,000
MNIST classes: ['0 - zero', '1 - one', '2 - two', '3 - three', '4 - four', '5 - five', '6 - six', '7 - seven', '8 - eight', '9 - nine']


In [5]:
# ── SVHN ───────────────────────────────────────────────────────────────────
svhn_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

# SVHN uses split='train' / split='test' instead of train=True/False
svhn_train = SVHN(root="../data", split="train", download=True, transform=svhn_transform)
svhn_test  = SVHN(root="../data", split="test",  download=True, transform=svhn_transform)

svhn_train_loader, svhn_val_loader, svhn_test_loader = make_loaders(
    svhn_train, svhn_test,
    loader_kwargs=loader_kwargs,
    val_fraction=VAL_FRACTION,
)

svhn_classes = [str(c) for c in sorted(set(svhn_train.labels.tolist()))]
print(f"SVHN classes: {svhn_classes}")

Dataset split — train: 58,606  val: 14,651  test: 26,032
SVHN classes: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']


## Experiment A — Train SimpleCNN on MNIST from scratch

MNIST is presented as pseudo-RGB (`[3, 32, 32]`). The model is trained from random initialisation.  
The best checkpoint (restored automatically by `fit`) is reused in Experiments B and C.

In [6]:
NUM_EPOCHS    = 50
LEARNING_RATE = 1e-3

model_mnist     = SimpleCNN(num_classes=10).to(device)
optimizer_mnist = optim.Adam(model_mnist.parameters(), lr=LEARNING_RATE)

wandb_kwargs_a = dict(
    entity="d7047e-group12",
    project="Lab0",
    name="SimpleCNN MNIST",
    tags=["Task 0.2.2", "MNIST"],
    config={
        "dataset": "MNIST",
        "optimizer": "Adam",
        "lr": LEARNING_RATE,
        "activation": "LeakyReLU",
        "epochs": NUM_EPOCHS,
        "batch_size": BATCH_SIZE,
    },
)

history_mnist = fit(
    model=model_mnist,
    optimizer=optimizer_mnist,
    criterion=nn.CrossEntropyLoss(),
    train_loader=mnist_train_loader,
    val_loader=mnist_val_loader,
    num_epochs=NUM_EPOCHS,
    wandb_kwargs=wandb_kwargs_a,
    log=LOG_WANDB,
)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: oscar-engelmark (d7047e-group12) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch  1/50 | train loss 0.2815, train acc 91.07% | val loss 0.0589, val acc 98.22%
Epoch  2/50 | train loss 0.0536, train acc 98.36% | val loss 0.0444, val acc 98.66%
Epoch  3/50 | train loss 0.0351, train acc 98.87% | val loss 0.0360, val acc 98.91%
Epoch  4/50 | train loss 0.0264, train acc 99.17% | val loss 0.0343, val acc 98.92%
Epoch  5/50 | train loss 0.0201, train acc 99.37% | val loss 0.0383, val acc 98.86%
Epoch  6/50 | train loss 0.0175, train acc 99.41% | val loss 0.0331, val acc 99.00%
Epoch  7/50 | train loss 0.0145, train acc 99.53% | val loss 0.0305, val acc 99.19%
Epoch  8/50 | train loss 0.0106, train acc 99.65% | val loss 0.0372, val acc 99.04%
Epoch  9/50 | train loss 0.0101, train acc 99.65% | val loss 0.0354, val acc 99.20%
Epoch 10/50 | train loss 0.0075, train acc 99.76% | val loss 0.0355, val acc 99.12%
Epoch 11/50 | train loss 0.0077, train acc 99.74% | val loss 0.0368, val acc 98.97%
Epoch 12/50 | train loss 0.0094, train acc 99.68% | val loss 0.0312, val acc

Training Accuracy,▁▇▇█████████████████████████████████████
Training Loss,█▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Validation Accuracy,▁▄▅▅▅▇▆▇▆▆▇▇▇▇▆▇▇▇▆▇▅▇▇▅▇███▇▆▇█████████
Validation Loss,█▄▂▃▂▃▂▂▃▁▁▂▂▃▂▃▃▄▄▄▃▃▅▅█▄▃▄▄▄▃▃▃▃▃▃▄▄▄▄
Training Accuracy,100
Training Loss,0.0
Validation Accuracy,99.36667
Validation Loss,0.04229



Restored best weights (val loss 0.0304)


In [7]:
_, mnist_test_acc = evaluate(
    model=model_mnist,
    test_loader=mnist_test_loader,
    criterion=nn.CrossEntropyLoss(),
    label="SimpleCNN — MNIST",
)

[SimpleCNN — MNIST] Test loss: 0.0249 | Test acc: 99.29%


## Experiment B — Zero-shot evaluation on SVHN

We evaluate the MNIST-trained model **directly** on the SVHN test set, without any further training.  
This shows how much knowledge transfers across the domain gap before any adaptation.

In [8]:
_, svhn_zeroshot_acc = evaluate(
    model=model_mnist,
    test_loader=svhn_test_loader,
    criterion=nn.CrossEntropyLoss(),
    label="SimpleCNN — SVHN (zero-shot from MNIST)",
)

[SimpleCNN — SVHN (zero-shot from MNIST)] Test loss: 2.0491 | Test acc: 34.42%


## Experiment C — Fine-tuning on SVHN (transfer learning)

We initialise a new `SimpleCNN` from the MNIST checkpoint and fine-tune the whole network on SVHN.  
Because both models share the same architecture and `in_channels=3`, all weights copy cleanly via `load_state_dict`.  
A slightly lower learning rate is used to avoid overwriting the transferred knowledge too aggressively.

In [9]:
# ── Initialise SVHN model from MNIST checkpoint ────────────────────────────
model_svhn = SimpleCNN(num_classes=10).to(device)
model_svhn.load_state_dict(copy.deepcopy(model_mnist.state_dict()))
print("All weights transferred successfully.")

All weights transferred successfully.


In [10]:
NUM_EPOCHS = 50
SVHN_LR    = 1e-4  # lower than MNIST LR to preserve transferred knowledge

optimizer_svhn = optim.Adam(model_svhn.parameters(), lr=SVHN_LR)

wandb_kwargs_c = dict(
    entity="d7047e-group12",
    project="Lab0",
    name="SimpleCNN SVHN (fine-tune from MNIST)",
    tags=["Task 0.2.2", "SVHN", "Transfer Learning"],
    config={
        "dataset": "SVHN",
        "pretrained_on": "MNIST",
        "optimizer": "Adam",
        "lr": SVHN_LR,
        "activation": "LeakyReLU",
        "epochs": NUM_EPOCHS,
        "batch_size": BATCH_SIZE,
    },
)

history_svhn = fit(
    model=model_svhn,
    optimizer=optimizer_svhn,
    criterion=nn.CrossEntropyLoss(),
    train_loader=svhn_train_loader,
    val_loader=svhn_val_loader,
    num_epochs=NUM_EPOCHS,
    wandb_kwargs=wandb_kwargs_c,
    log=LOG_WANDB,
)

Epoch  1/50 | train loss 1.2786, train acc 58.99% | val loss 0.8826, val acc 72.81%
Epoch  2/50 | train loss 0.7578, train acc 77.29% | val loss 0.6651, val acc 80.21%
Epoch  3/50 | train loss 0.6103, train acc 82.08% | val loss 0.5728, val acc 83.36%
Epoch  4/50 | train loss 0.5280, train acc 84.62% | val loss 0.5124, val acc 85.15%
Epoch  5/50 | train loss 0.4740, train acc 86.24% | val loss 0.4791, val acc 85.81%
Epoch  6/50 | train loss 0.4340, train acc 87.30% | val loss 0.4500, val acc 86.93%
Epoch  7/50 | train loss 0.4039, train acc 88.22% | val loss 0.4274, val acc 87.63%
Epoch  8/50 | train loss 0.3768, train acc 89.05% | val loss 0.4192, val acc 87.93%
Epoch  9/50 | train loss 0.3553, train acc 89.74% | val loss 0.3928, val acc 88.86%
Epoch 10/50 | train loss 0.3351, train acc 90.40% | val loss 0.3820, val acc 88.85%
Epoch 11/50 | train loss 0.3176, train acc 90.80% | val loss 0.3755, val acc 89.08%
Epoch 12/50 | train loss 0.3035, train acc 91.33% | val loss 0.3638, val acc

Training Accuracy,▁▄▅▅▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇█████████████████
Training Loss,█▅▄▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Validation Accuracy,▁▄▆▆▆▇▇▇▇▇██████████████████████████████
Validation Loss,█▅▄▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▂▂▂▂▂▂▂▃▃▃
Training Accuracy,99.46081
Training Loss,0.03119
Validation Accuracy,90.54672
Validation Loss,0.46977



Restored best weights (val loss 0.3297)


In [11]:
_, svhn_test_acc = evaluate(
    model=model_svhn,
    test_loader=svhn_test_loader,
    criterion=nn.CrossEntropyLoss(),
    label="SimpleCNN — SVHN (fine-tuned from MNIST)",
)

[SimpleCNN — SVHN (fine-tuned from MNIST)] Test loss: 0.3661 | Test acc: 90.12%


## Summary

In [12]:
print("===== Final Results =====")
print(f"Exp A — MNIST             : best val acc {max(history_mnist['val_acc']):.2f}%  |  test acc {mnist_test_acc:.2f}%")
print(f"Exp B — SVHN zero-shot    :                      |  test acc {svhn_zeroshot_acc:.2f}%")
print(f"Exp C — SVHN fine-tuned   : best val acc {max(history_svhn['val_acc']):.2f}%  |  test acc {svhn_test_acc:.2f}%")

===== Final Results =====
Exp A — MNIST             : best val acc 99.38%  |  test acc 99.29%
Exp B — SVHN zero-shot    :                      |  test acc 34.42%
Exp C — SVHN fine-tuned   : best val acc 91.10%  |  test acc 90.12%


## Brief Discussion

Both MNIST and SVHN are 10-class digit recognition tasks, making MNIST a natural source domain. The shared task structure means low-level features (edge detectors, stroke patterns) learned on MNIST can generalise to SVHN.

**Experiment B** (zero-shot) reveals how much structure transfers with no adaptation at all. A significant **domain gap** remains — MNIST digits are clean, centred, and synthetically simple, while SVHN images contain colour, clutter, and multiple overlapping digits — so zero-shot accuracy is expected to be well below the MNIST test accuracy.

**Experiment C** (fine-tuning) closes most of that gap. Starting from the MNIST weights rather than random initialisation gives the optimiser a better starting point, which typically leads to faster convergence and a higher final accuracy than training on SVHN from scratch.

By converting MNIST to pseudo-RGB all weights — including the first conv layer — transfer without modification, which is preferable to discarding the first layer or converting SVHN to grayscale (which would lose colour information useful for digit segmentation).